In [10]:
from f_0_dirs import get_data_dirs
dirs = get_data_dirs()

# Iterate over the attributes of the DirPaths object
for attr in dir(dirs):
    if attr.startswith('_'):
        continue
    if callable(getattr(dirs, attr)):
        continue
    print(f"{attr}: {getattr(dirs, attr)}")

data_dir: D:\FAME - LN dataset\Dropbox\fame_clean\1_FAME_raw_data\2025.07.30
output_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\output
raw_data_dir: D:\FAME - LN dataset\Dropbox\fame_clean\1_FAME_raw_data\2025.02
root_data_dir: D:\FAME - LN dataset\Dropbox\fame_clean
root_dir: C:\Users\lazyst\Files\ucl\Dissertation
work_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\src


# 1. [read] from FAME

### Categorise raw files

This script resolves the project data paths, scans the raw-data folder, builds a nested dictionary of file metadata by company and file category, and writes it to JSON.  
`get_data_dirs()` defines the working directories  
`build_raw_file_dict()` performs the recursive traversal and file collection through its helper functions.  

In [11]:
from flask import json
import pandas as pd

from f_1_traverse import build_raw_file_dict
pd.options.mode.chained_assignment = None  # default='warn'

if dirs.raw_data_dir is None:
	raise ValueError("raw_data_dir is None. Please check your .env file and ensure RAW_DATA_DIR is set correctly.")

raw_file_dict = build_raw_file_dict(dirs.raw_data_dir)
with open(dirs.output_dir / "raw_file_dict.json", "w") as f:
    json.dump(raw_file_dict, f, indent=4)
    print(f"✅ Successfully built raw file dictionary and saved to: {dirs.output_dir / 'raw_file_dict.json'}")

Traversing industry directory: 01, 02, 03, 05, 06, 07, 08, 09, 10, 11, 12, 13, 14, 15, 16
17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31
32, 33, 35, 36, 37, 38, 39, 41, 42, 43, 45, 46, 47, 49, 50
51, 52, 53, 55, 56, 58, 59, 60, 61, 62, 63, 64, 65, 66, 68
69, 70, 71, 72, 73, 74, 75, 77, 78, 79, 80, 81, 82, 84, 85
86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99
✅ Successfully built raw file dictionary and saved to: C:\Users\lazyst\Files\ucl\Dissertation\build\output\raw_file_dict.json


### Process each excel file in the raw file dictionary

In [12]:
import pandas as pd

# import xlsx file from input/raw_properties.xlsx to load as a schema
# Declare types that the schema_source df has colums ["from_raw", "key", "type", "fuzzy_mapping", "in_raw_data", "keep", "in_ln_set", "description"]
schema_path = dirs.root_dir / "build" / "input" / "raw_properties.xlsx"
schema_source = pd.read_excel(schema_path, sheet_name="raw_properties", engine="openpyxl")

# Schema for raw inputs is rows where from_raw is not blank
schema_raw: pd.DataFrame = schema_source[schema_source["from_raw"].notna()]

# Take our master schema and turn it into a helpful mapping of fuzzy column names to schema column names
# Regardless of what the raw source is. We'll filter it later
schema_fixed_fuzzy_df: pd.DataFrame = schema_source[["key", "fuzzy_mapping"]]
schema_fixed_fuzzy_df["fml"] = schema_fixed_fuzzy_df["fuzzy_mapping"].str.split('\n')

# Print any rows where fml has more than one element
for index, row in schema_fixed_fuzzy_df.iterrows():
    if row["fml"] is None:
        print(f"Row {index} has fml that is None: {row['key']}")
    elif type(row["fml"]) is not list:
        print(f"Row {index} has fml that is not a list: {row['key']}")
    elif len(row["fml"]) > 1:
        print(f"Row {index} has more than one fuzzy mapping: {row['fml']}")

Row 37 has more than one fuzzy mapping: ['Strategy,  organization and policy', 'Strategy, organization and policy']
Row 49 has fml that is not a list: has_ptaddress
Row 50 has fml that is not a list: has_ptaddress_latlong
Row 51 has fml that is not a list: is_public
Row 52 has fml that is not a list: has_company_branch_mismatch
Row 53 has fml that is not a list: industry_code
Row 54 has fml that is not a list: file_code


# 2. [write] To duck schemas

### Define duck schemas

In [ ]:
import pandas as pd
# Import ibis-framework
import ibis
# pip install 'ibis-framework[duckdb,geospatial]'

print("Path:", ibis.__file__)
print("Version:", getattr(ibis, "__version__", "No version found"))
pd.options.mode.chained_assignment = None  # default='warn'

db_path = dirs.output_dir / "fame_data.duckdb"

# Fixed schema
# schema_fixed is a df of schema_source where values in column "keep" are "fixed" or "all"
schema_fixed: pd.DataFrame = schema_source[schema_source["keep"].isin(["fixed", "all"])]
schema_fixed_dict: dict[str, str] = dict(zip(schema_fixed["key"], schema_fixed["type"]))
schema_fixed_ibis: ibis.Schema = ibis.schema(schema_fixed_dict)
schema_fixed_names: set[str] = set(schema_fixed_ibis.keys())

schema_derived: pd.DataFrame = schema_source[schema_source["keep"].isin(["derived", "all"])]
schema_derived_dict: dict[str, str] = dict(zip(schema_derived["key"], schema_derived["type"]))
schema_derived_ibis: ibis.Schema = ibis.schema(schema_derived_dict)
schema_derived_names: set[str] = set(schema_derived_ibis.keys())

# build a schema_yearly df with columns registered_number, fame_key, year, value properties, with types str, str, int, float
schema_yearly: pd.DataFrame = pd.DataFrame({
    "key": ["registered_number", "fame_key", "year", "value"],
    "type": ["string", "string", "int64", "float64"]
})
schema_yearly_dict: dict[str, str] = dict(zip(schema_yearly["key"], schema_yearly["type"]))
schema_yearly_ibis: ibis.Schema = ibis.schema(schema_yearly_dict)
schema_yearly_names: set[str] = set(schema_source[schema_source["keep"] == "yearly" and schema_source["from_raw"].notna()]["key"].tolist())

# 4. Execute the table creation using the Ibis schema
try:
    
    # 2. Connect to DuckDB using Ibis
    con = ibis.duckdb.connect(str(db_path))
    print(f"Initializing DuckDB via Ibis at: {db_path}")
    # overwrite=True prevents errors if the script is run multiple times during setup
    con.create_table("fame_fixed", schema=schema_fixed_ibis, overwrite=True)
    print("✅ Successfully created Ibis schema for 'fame_fixed'.")
    print("\nTable Schema Verification:")
    print(con.table("fame_fixed").schema())

    con.create_table("fame_derived", schema=schema_derived_ibis, overwrite=True)
    print("✅ Successfully created Ibis schema for 'fame_derived'.")
    print("\nTable Schema Verification:")
    print(con.table("fame_derived").schema())

    con.create_table("fame_yearly", schema=schema_yearly_ibis, overwrite=True)
    print("✅ Successfully created Ibis schema for 'fame_yearly'.")
    print("\nTable Schema Verification:")
    print(con.table("fame_yearly").schema())

except Exception as e:
    print(f"❌ Error creating table: {e}")

Path: c:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\Lib\site-packages\ibis\__init__.py
Version: 12.0.0
Initializing DuckDB via Ibis at: C:\Users\lazyst\Files\ucl\Dissertation\build\output\fame_data.duckdb
✅ Successfully created Ibis schema for 'fame_fixed'.

Table Schema Verification:
ibis.Schema {
  company_name                       string
  registered_number                  string
  ticker_symbol                      string
  primary_trading_address            string
  primary_trading_address_latitude   string
  primary_trading_address_longitude  string
  branch_name                        string
  primary_uk_sic_2007_code           int64
  primary_uk_sic_2007_description    string
  latest_accounts_date               date
  no_of_available_years              int64
}
✅ Successfully created Ibis schema for 'fame_derived'.

Table Schema Verification:
ibis.Schema {
  registered_number            string
  has_ptaddress                boolean
  has_ptaddress_latlong        boo

### Load, modify and write imported file schema

In [ ]:
from flask import json
from f_1_traverse import RawFileDict
from f_2_check import drop_duplicate_columns, check_df_matches_schema, handle_excel_dates
from f_2_modify import coerce_df_dates_from_schema
import random

# Traverse the raw_file_dict.json file to get each Excel filepath
# declare raw_file_dict as a RawFileDict type
raw_file_dict: RawFileDict | None = None
with open(dirs.output_dir / "raw_file_dict.json", "r") as f:
    raw_file_dict = json.load(f)
if raw_file_dict is None:
    raise ValueError("❌ Error: raw_file_dict.json is empty or not found.")
if dirs.raw_data_dir is None:
    raise ValueError("❌ Error: raw_data_dir is None. Please check your .env file and ensure RAW_DATA_DIR is set correctly.")

# Handle yearly variables
start_year = 2006
end_year = 2025

process_count = 0

# We want one big dataframe, which we will merge all the data into for now
# df_fixed = pd.DataFrame(columns=list(schema_fixed.columns))
ind_keys = raw_file_dict.keys()
ind_shuffled = list(ind_keys)
random.shuffle(ind_shuffled)

for ind, obj in raw_file_dict.items():
    for property, arr in obj.items():

        # DECLARE THE SCHEMA for this input
        schema_raw_fuzzy_filtered = schema_raw[schema_raw["from_raw"].isin([property, 'all'])]
        schema_raw_fuzzy_filtered["fml"] = schema_raw_fuzzy_filtered["fuzzy_mapping"].str.split('\n')
        schema_raw_fuzzy_mapping: dict[str, str] = dict(zip(
            schema_raw_fuzzy_filtered["key"],
            schema_raw_fuzzy_filtered["fml"]
        ))
        schema_raw_fuzzy_col_map = {
            raw_name: schema_name for schema_name,
            raw_names in schema_raw_fuzzy_mapping.items() for raw_name in raw_names
        }

        files_shuffled = arr.copy()
        random.shuffle(files_shuffled)
        for [file_name, file_path] in files_shuffled:
            
            file_code = " ".join(file_name.split()[2:]).replace(".xlsx", "")    # Export 22_01_2026 09_39 1.xlsx -> 09_39 1

            if process_count >= 5:
                break

            # LOAD
            df_raw = pd.read_excel(file_path, sheet_name='Results', header=0, dtype={
                "registered_number": str                                            # "Leading Zeros" Trap. Pandas accidentally processes
            })                                                                      #   registered number as int64 when reading the file, which will chop zeros.
            df_raw.drop(df_raw.columns[0], axis=1, inplace=True)                    # Drop column A (blank in raw data)
            df_raw = drop_duplicate_columns(df_raw)                                 # Remove duplicate columns (quirk of some files)
            df_raw.rename(columns=schema_raw_fuzzy_col_map, inplace=True)           # Rename according to our mapping
            df_raw = df_raw[df_raw['registered_number'].notna()]                    # Drop rows with NaN registered number

            # FIX: for yearly variables of the form 'my_key 2005', we need to extract the year and somehow store it
            # FIX: then fuzzy match the remainder of the column name to the schema like all the other tables
            # FIX: saving that year for later so we can use it in an unzipped format for the yearly table

            # switch case on property      
            try:

                if property in ["a1_ID"]:
                    # FILTER
                    df_raw = df_raw[df_raw['no_of_available_years'] != 0]                   # Drop rows with 0 in 'no_of_available_years'
                    if 'ro_country' in df_raw.columns:
                        df_raw = df_raw[df_raw['ro_country'] != "Republic of Ireland"]      # Drop rows where ro_country is specifically defined as "Republic of Ireland"
                    df_raw = handle_excel_dates(df_raw)                                     # Handle any Excel date serials to datetime
                    try:
                        check_df_matches_schema(schema_raw_fuzzy_mapping, df_raw)           # Check if the DataFrame matches the input
                    except ValueError as e:
                        print(f"⚠️ Mismatch in raw data validation for file {ind}/{property}/{file_name})")
                        print("Warning:", e)

                if property in ["a1_ID", "a5_misc"]: 
                    # MODIFY
                    # Filter to only columns that exist in the target schema and load to ibis
                    # Load up data tables and merge them
                    table_fame_fixed: ibis.expr.types.Table     = con.table("fame_fixed")
                    df_fixed_one_dates = pd.DataFrame(columns=list(schema_fixed_ibis.keys()))
                    df_fixed_one = df_raw[[col for col in df_raw.columns if col in schema_fixed_names]]
                    df_fixed_one = df_fixed_one.reindex(columns=list(schema_fixed_names), fill_value=pd.NA) # type: ignore
                    df_fixed_one_dates = coerce_df_dates_from_schema(schema_fixed_ibis, df_fixed_one)
                    table_t_raw = ibis.memtable(df_fixed_one_dates)

                    # 1. Cast types dynamically
                    table_fixed_type_casts = {
                        col: table_t_raw[col].cast(schema_fixed_ibis.fields[col])
                        for col in schema_fixed_names
                            if col in table_t_raw.columns # and not table_t_raw[col].type().equals(schema_fixed_ibis.fields[col])
                    }
                    table_t_raw_cast = table_t_raw.mutate(**table_fixed_type_casts).select(schema_fixed_names)

                    # 2. UPSERT LOGIC: keep existing records and add new records as new rows, and then after update with the union
                    table_merged_fixed_raw = table_fame_fixed.anti_join(
                        table_t_raw_cast,
                        "registered_number"
                    )
                    table_updated_fixed_raw = table_merged_fixed_raw.union(table_t_raw_cast)
                    con.create_table("fame_fixed", table_updated_fixed_raw, overwrite=True)
                    print(f"✅ Successfully processed fixed table from: {ind}/{property}/{file_name}")

                if property in ["a1_ID"]:
                    # DERIVE
                    # Refresh connection to fixed_fame table
                    # And then add derived column properties
                    table_fame_fixed: ibis.expr.types.Table     = con.table("fame_fixed")
                    table_fame_derived: ibis.expr.types.Table   = con.table("fame_derived")
                    table_joined: ibis.expr.types.Table         = table_fame_fixed.left_join(
                        table_fame_derived,
                        "registered_number"
                    ).mutate(
                        has_ptaddress = table_fame_fixed.primary_trading_address.notnull(),
                        has_ptaddress_latlong = table_fame_fixed.primary_trading_address_latitude.notnull()
                            & table_fame_fixed.primary_trading_address_longitude.notnull(),
                        is_public = table_fame_fixed.ticker_symbol.notnull(),
                        has_company_branch_mismatch = table_fame_fixed.company_name != table_fame_fixed.branch_name,
                        industry_code = ibis.literal(ind),
                        file_code = ibis.literal(file_code)
                    )
                    table_updated_derived = table_joined.select(
                        table_fame_derived.columns
                    )
                    con.create_table("fame_derived", table_updated_derived, overwrite=True)
                    print(f"✅ Successfully processed derived table from: {ind}/{property}/{file_name}")

                if property in ["a2_key_finance", "a3_assets", "a4_profits"]:

                    
            except Exception as e:
                print(f"❌ Error processing file {ind}/{property}/{file_name}")
                print("Error:", e)
                print("")

            process_count += 1

# Drop all temporary tables
for table_name in con.list_tables():
    # continue if the table name is fame_fixed or fame_derived
    if table_name in ["fame_fixed", "fame_derived", "fame_yearly"]:
        continue
    try:
        con.drop_table(table_name)
        print(f"✅ Successfully dropped table {table_name}.")
    except:
        pass

# Output the head of the fame_fixed and fame_derived tables to verify the data
print("\nHead of fame_derived table:")
print(con.table("fame_derived").execute().head())
print("\nHead of fame_fixed table:")
print(con.table("fame_fixed").execute().head())
print(con.table("fame_fixed").count().execute())

⚠️ Converted Excel date serials to datetime for column 'latest_accounts_date'
⚠️ Coerced column 'latest_accounts_date' to datetime based on schema type 'date'
✅ Successfully processed fixed table from: 01/a1_ID/Export 15_02_2025 12_33.xlsx
✅ Successfully processed derived table from: 01/a1_ID/Export 15_02_2025 12_33.xlsx
⚠️ Converted Excel date serials to datetime for column 'latest_accounts_date'
⚠️ Coerced column 'latest_accounts_date' to datetime based on schema type 'date'
✅ Successfully processed fixed table from: 01/a1_ID/Export 15_02_2025 12_33 1.xlsx
✅ Successfully processed derived table from: 01/a1_ID/Export 15_02_2025 12_33 1.xlsx
⚠️ Converted Excel date serials to datetime for column 'latest_accounts_date'
⚠️ Coerced column 'latest_accounts_date' to datetime based on schema type 'date'
✅ Successfully processed fixed table from: 01/a1_ID/Export 15_02_2025 12_34 1.xlsx
✅ Successfully processed derived table from: 01/a1_ID/Export 15_02_2025 12_34 1.xlsx
✅ Successfully processe

# 3. [view] resulting DB for inspection

### basic tables overview

In [17]:
# List tables in the DuckDB database as an .md file in /tmp
# Give me the head of all tables
# Ensure they are nicely formatted with headers so I can easily see what's going on
out_file = dirs.root_dir / "build" / "tmp" / "duckdb_tables.md"
with open(out_file, "w") as f:
    tables = con.list_tables()
    f.write("# Tables in DuckDB database\n\n")
    for table in tables:
        f.write(f"## {table}\n\n")
        f.write(f"### Number of rows: {con.table(table).count().execute()}\n\n")
        f.write(f"### Schema:\n\n```\n{con.table(table).schema()}\n```\n\n")
        f.write(f"### Head of table:\n\n```\n{con.table(table).execute().head()}\n```\n\n")
print(f"✅ Successfully listed tables and their heads in: {out_file}")

# Check for any duplicate registered_number values in fame_fixed and fame_derived
for table_name in ["fame_fixed", "fame_derived"]:
    table_df = con.table(table_name).execute()
    duplicate_registered_numbers = table_df[table_df.duplicated(subset=["registered_number"], keep=False)]
    if not duplicate_registered_numbers.empty:
        print(f"⚠️ Warning: Duplicate registered_number values found in {table_name}:")
        print(duplicate_registered_numbers)
    else:
        print(f"No duplicates found in {table_name}.")

✅ Successfully listed tables and their heads in: C:\Users\lazyst\Files\ucl\Dissertation\build\tmp\duckdb_tables.md
No duplicates found in fame_fixed.
No duplicates found in fame_derived.


In [19]:
# Dump the first 500 rows of df_raw, fame_derived, and famed_fixed to a single .xlsx file in /tmp
# Using those as different sheet names
# Get fame_derived and fame_fixed from their ibis tables
out_file_raw = dirs.root_dir / "build" / "tmp" / "df_raw_head.xlsx"
df_raw_head = df_raw.head(500)
df_derived_head = con.table("fame_derived").execute().head(500)
df_fixed_head = con.table("fame_fixed").execute().head(500)
with pd.ExcelWriter(out_file_raw, engine='openpyxl') as writer:
    df_raw_head.to_excel(writer, sheet_name='df_raw', index=False)
    df_derived_head.to_excel(writer, sheet_name='fame_derived', index=False)
    df_fixed_head.to_excel(writer, sheet_name='fame_fixed', index=False)
print(f"✅ Successfully dumped the first 500 rows of df_raw to: {out_file_raw}")

✅ Successfully dumped the first 500 rows of df_raw to: C:\Users\lazyst\Files\ucl\Dissertation\build\tmp\df_raw_head.xlsx


# 4. Geospatial processing

In [ ]:
import ibis
from f_3_spatial import convert_dms_to_decimal
from f_0_dirs import get_data_dirs

dirs = get_data_dirs()
db_path = dirs.output_dir / "fame_data.duckdb"
# con.raw_sql("INSTALL spatial; LOAD spatial;")

# DERIVE
table_fame_fixed: ibis.expr.types.Table   = con.table("fame_fixed")
table_fame_derived: ibis.expr.types.Table = con.table("fame_derived")
table_joined = table_fame_fixed.left_join(table_fame_derived, "registered_number")

# Assume you load a free UK Postcode to Lat/Lon lookup CSV into DuckDB
# table_postcode_lookup = con.table("uk_postcodes") 

# 2. Mutate Hierarchy & Coords
table_mutated = table_joined.mutate(
    
    # --- ADDRESS HIERARCHY ---
    # Returns the first option that isn't Null
    best_full_address = ibis.coalesce(
        table_fame_fixed.primary_trading_address,
        table_fame_fixed.ro_address,
        # Fallback: concatenate the separate lines if the above are null
        ibis.literal(", ").join(
            ibis.array([
                table_fame_fixed.ro_address_line_1, 
                table_fame_fixed.ro_address_line_2, 
                table_fame_fixed.ro_address_postcode
            ]).filter(lambda x: x.notnull()) # Only join non-null lines
        )
    ),
    
    # --- GEOSPATIAL HIERARCHY ---
    # 1. Parse FAME's DMS strings into pure decimal floats
    fame_lat_dec = convert_dms_to_decimal(table_fame_fixed.primary_trading_address_latitude),
    fame_lon_dec = convert_dms_to_decimal(table_fame_fixed.primary_trading_address_longitude),
    
    # 2. (Optional Future Step) If you joined a postcode lookup table, 
    # you would include its lat/lon here as a fallback
    # lookup_lat = table_postcode_lookup.latitude,
    
    # 3. Store the best available coordinates
    best_latitude = ibis.coalesce(
        convert_dms_to_decimal(table_fame_fixed.primary_trading_address_latitude),
        # lookup_lat
    )
)

# 3. Select Derived Schema Columns and Save
table_derived = table_mutated.select(schema_derived_names)
con.create_table("fame_derived", table_derived, overwrite=True)